# Transformation to SQL query

In [0]:
query = """
SELECT
    crm.customer_id,
    crm.customer_key,
    crm.first_name,
    crm.last_name,
    crm.marital_status,
    COALESCE(erp.gender, crm.gender) AS gender,
    erp.birth_date,
    loc.country,
    crm.create_date
FROM silver.crm_customers crm
LEFT JOIN silver.erp_customers erp
    ON crm.customer_key = erp.customer_key
LEFT JOIN silver.erp_locations loc
    ON crm.customer_key = loc.customer_key
"""

df = spark.sql(query)

# Preview

In [0]:
df.display()
print("Row count:", df.count())

# Duplicate customer_key check

In [0]:
duplicate_count = (
    df.groupBy("customer_key")
      .count()
      .filter("count > 1")
      .count()
)

print("Duplicate customer_key:", duplicate_count)

if duplicate_count > 0:
    raise Exception("Duplicate customer_key in dim_customers!")

#Write it to Gold Table

In [0]:
df.write.mode("overwrite").format("delta").saveAsTable("workspace.gold.dim_customers")


## Sanity check

In [0]:
%sql
SELECT *
FROM gold.dim_customers
LIMIT(20)


